<a href="https://colab.research.google.com/github/HisameOgasahara/irodori_test/blob/main/Irodori_v4_1_RF_Yuina_LoRA_Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 유이나 · Irodori-TTS v4.1 RF LoRA

**GPU 런타임에서 위에서부터 실행하세요.** `yuina_dataset.zip`을 Google Drive에 올리고 1번 셀의 `dataset_zip_path`를 입력합니다.
ZIP 안의 `yuina/captions.csv`와 `yuina/wav/`를 읽어 일본어 대사를 학습합니다. 기존 데이터는 수정하지 않습니다.

- 기본 모델: **Aratako/Irodori-TTS-v4.1-Small (RF)**. MF/양자화 모델을 학습에 사용하지 않습니다.
- 데이터 복사본, latent, 모델 캐시, 체크포인트, 추론 WAV는 모두 **`/content/irodori_work`**에 저장됩니다. Drive에 학습 결과를 쓰는 셀은 없습니다.
- **코랩 런타임이 초기화되면 체크포인트가 사라집니다.** 보관할 결과는 마지막 다운로드 셀로 PC에 받으세요.
- A100/L4처럼 BF16을 지원하는 GPU를 권합니다. T4는 공식 코드에 FP16 학습이 없어 FP32로 실행하며 메모리 부족/느린 학습이 발생할 수 있습니다. GPU 실학습은 이 노트북 제작 환경에서 검증하지 않았습니다.
- 기본값은 1,000 step 본 학습입니다. 먼저 확인하려면 7번 셀의 `smoke_test`를 체크해 2 step 시험 후 해제하고 다시 실행하세요. 1,000 step은 탐색용 시작값이며 품질을 보장하는 값이 아닙니다.

공식 소스: [Irodori-TTS](https://github.com/Aratako/Irodori-TTS), [RF LoRA 설정](https://github.com/Aratako/Irodori-TTS/blob/main/configs/train_v4_small_lora.yaml), [Cloudflare Quick Tunnel 제약](https://developers.cloudflare.com/cloudflare-one/networks/connectors/cloudflare-tunnel/do-more-with-tunnels/trycloudflare/).
코드 commit `89f9d8fbd4d51ea019867ee1197725ede1df13c5`와 모델 revision을 고정하고 공식 `uv.lock`을 사용합니다.


In [6]:
#@title 1. 데이터 ZIP 경로와 학습 설정
dataset_zip_path = "/content/drive/MyDrive/yuina_voice/yuina_dataset.zip" #@param {type:"string"}
run_name = "yuina_rf_lora" #@param {type:"string"}
max_steps = 1000 #@param {type:"integer"}
batch_size = 1 #@param {type:"integer"}
gradient_accumulation_steps = 8 #@param {type:"integer"}
learning_rate = 0.0001 #@param {type:"number"}
save_every = 250 #@param {type:"integer"}
lora_rank = 16 #@param {type:"integer"}
min_audio_seconds = 0.5 #@param {type:"number"}
max_audio_seconds = 20.0 #@param {type:"number"}
reference_max_seconds = 10.0 #@param {type:"number"}
seed = 42 #@param {type:"integer"}

from pathlib import Path
import os, re, json, subprocess, shutil, hashlib, time, signal, urllib.request
assert re.fullmatch(r"[A-Za-z0-9_-]+", run_name), "run_name은 영문/숫자/_/-만 사용하세요."
assert max_steps >= 2 and batch_size >= 1 and gradient_accumulation_steps >= 1
assert save_every >= 1 and lora_rank >= 1 and 0 < learning_rate < 1
assert 0 < min_audio_seconds < max_audio_seconds <= 30
assert 1 <= reference_max_seconds <= 120
WORK = Path('/content/irodori_work')
REPO = WORK / 'Irodori-TTS'
RUN = WORK / 'runs' / run_name
DATA = RUN / 'data'
OUTPUT = RUN / 'checkpoints'
DRIVE = Path('/content/drive')
CODE_REVISION = '89f9d8fbd4d51ea019867ee1197725ede1df13c5'
MODEL_ID = 'Aratako/Irodori-TTS-v4.1-Small'
MODEL_REVISION = '2b28324dc263ed5e6638b3cf3dd94c82ead07b4b'
for p in (WORK, RUN, DATA, OUTPUT):
    p.mkdir(parents=True, exist_ok=True)
    assert p.resolve().is_relative_to(Path('/content'))
    assert not p.resolve().is_relative_to(DRIVE.resolve())
os.environ.update({
    'HF_HOME': str(WORK / 'cache/huggingface'),
    'UV_CACHE_DIR': str(WORK / 'cache/uv'),
    'TORCH_HOME': str(WORK / 'cache/torch'),
    'GRADIO_TEMP_DIR': str(WORK / 'gradio_tmp'),
    'GRADIO_ANALYTICS_ENABLED': 'False',
    'TOKENIZERS_PARALLELISM': 'false',
})

def run(args, *, cwd=None, log=None, env=None):
    """Stream subprocess output and stop the child on notebook interruption."""
    handle = open(log, 'a', encoding='utf-8') if log else None
    proc = subprocess.Popen([str(a) for a in args], cwd=cwd, env=env,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, start_new_session=True)
    try:
        for line in proc.stdout:
            print(line, end='')
            if handle:
                handle.write(line)
                handle.flush()
        if proc.wait():
            raise RuntimeError(f'실행 실패 (exit={proc.returncode}). 위 로그를 확인하세요.')
    except BaseException:
        if proc.poll() is None:
            os.killpg(proc.pid, signal.SIGTERM)
            try:
                proc.wait(timeout=15)
            except subprocess.TimeoutExpired:
                os.killpg(proc.pid, signal.SIGKILL)
                proc.wait()
        raise
    finally:
        proc.stdout.close()
        if handle:
            handle.close()

def write_script(name, source):
    path = RUN / name
    path.write_text(source, encoding='utf-8')
    return path

print('체크포인트 저장 위치:', OUTPUT)


체크포인트 저장 위치: /content/irodori_work/runs/yuina_rf_lora/checkpoints


In [2]:
#@title 2. 공식 의존성 설치 (첫 실행은 수 분 걸립니다)
if shutil.which('nvidia-smi') is None:
    raise RuntimeError('런타임 → 런타임 유형 변경 → GPU를 선택하세요.')
run(['nvidia-smi'])
run(['apt-get', '-qq', 'update'])
run(['apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libsndfile1', 'git', 'build-essential'])
uv = shutil.which('uv') or str(Path.home() / '.local/bin/uv')
if not Path(uv).exists():
    installer = WORK / 'uv-install.sh'
    urllib.request.urlretrieve('https://astral.sh/uv/install.sh', installer)
    run(['sh', installer])
if not REPO.exists():
    run(['git', 'clone', 'https://github.com/Aratako/Irodori-TTS.git', REPO])
run(['git', '-C', REPO, 'checkout', '--detach', CODE_REVISION])
run([uv, 'python', 'install', '3.12'])
run([uv, 'sync', '--frozen', '--extra', 'cu128', '--python', '3.12'], cwd=REPO)
PY = REPO / '.venv/bin/python'
probe = write_script('probe_gpu.py', """import json, torch
assert torch.cuda.is_available(), "CUDA를 사용할 수 없습니다."
p = torch.cuda.get_device_properties(0)
info = {"name": p.name, "vram_gb": round(p.total_memory / 2**30, 1),
        "precision": "bf16" if torch.cuda.is_bf16_supported() else "fp32"}
print(json.dumps(info))
""")
GPU = json.loads(subprocess.check_output([str(PY), str(probe)], cwd=REPO, text=True))
print(GPU)
if GPU['precision'] == 'fp32':
    print('이 GPU는 FP32 학습을 사용합니다. OOM 발생 시 L4/A100 런타임을 권합니다.')
print('남은 로컬 디스크: %.1f GiB' % (shutil.disk_usage(WORK).free / 2**30))


Tue Sep 22 05:06:34 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

**Drive 사용 범위:** 아래 셀은 ZIP을 로컬로 복사한 후 Drive 연결을 해제합니다. 원본 ZIP과 기존 WAV는 덮어쓰지 않습니다.
동일 ZIP은 로컬 복사본을 재사용하며, 다른 ZIP을 사용하려면 `run_name`을 바꾸세요.


In [7]:
#@title 3. Drive의 ZIP 읽기 → 로컬 복사 → 안전하게 압축 해제
from google.colab import drive
from pathlib import PurePosixPath
import zipfile, stat
drive.mount(str(DRIVE))
source = Path(dataset_zip_path).expanduser().resolve()
if not source.is_relative_to(DRIVE.resolve()) or not source.is_file() or source.suffix.lower() != '.zip':
    raise ValueError('Google Drive에 있는 ZIP의 전체 경로를 입력하세요.')
LOCAL_ZIP = RUN / 'dataset.zip'
def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()
source_hash = sha256(source)
if LOCAL_ZIP.exists():
    if sha256(LOCAL_ZIP) != source_hash:
        raise RuntimeError('이 run_name에는 다른 ZIP이 있습니다. run_name을 바꾸세요.')
else:
    if shutil.disk_usage(WORK).free < source.stat().st_size * 2:
        raise RuntimeError('ZIP 복사용 로컬 디스크 공간이 부족합니다.')
    temporary = LOCAL_ZIP.with_suffix('.copying')
    shutil.copyfile(source, temporary)
    if sha256(temporary) != source_hash:
        raise RuntimeError('ZIP 복사 검증 실패. 셀을 다시 실행하세요.')
    temporary.rename(LOCAL_ZIP)
drive.flush_and_unmount()
EXTRACT = RUN / 'dataset'
stamp = RUN / 'dataset.sha256'
if stamp.exists() and stamp.read_text().strip() == source_hash and EXTRACT.is_dir():
    print('검증된 압축 해제본 재사용:', EXTRACT)
else:
    if EXTRACT.exists():
        raise RuntimeError('완료되지 않은 압축 해제 폴더가 있습니다. 새 run_name을 사용하세요.')
    with zipfile.ZipFile(LOCAL_ZIP) as z:
        infos = z.infolist()
        names = set()
        for info in infos:
            raw = info.filename
            p = PurePosixPath(raw)
            if (p.is_absolute() or '..' in p.parts or chr(92) in raw or ':' in raw
                    or stat.S_ISLNK(info.external_attr >> 16) or raw in names):
                raise ValueError(f'허용되지 않는 ZIP 항목: {raw}')
            names.add(raw)
            if not (EXTRACT / raw).resolve().is_relative_to(EXTRACT.resolve()):
                raise ValueError('ZIP 경로가 압축 해제 폴더를 벗어납니다.')
        required = sum(i.file_size for i in infos)
        if required > shutil.disk_usage(WORK).free - 5 * 2**30:
            raise RuntimeError('압축 해제 후 여유 공간 5 GiB를 확보할 수 없습니다.')
        bad = z.testzip()
        if bad:
            raise ValueError(f'ZIP CRC 오류: {bad}')
        EXTRACT.mkdir()
        z.extractall(EXTRACT)
    stamp.write_text(source_hash)
matches = list(EXTRACT.rglob('captions.csv'))
if len(matches) != 1:
    raise ValueError(f'ZIP 안에 captions.csv가 정확히 하나 있어야 합니다: {matches}')
CAPTIONS = matches[0]
print('대사 목록:', CAPTIONS)
print('Drive 연결 해제 완료. 이후 산출물은 /content에 저장됩니다.')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
대사 목록: /content/irodori_work/runs/yuina_rf_lora/dataset/yuina/captions.csv
Drive 연결 해제 완료. 이후 산출물은 /content에 저장됩니다.


In [8]:
#@title 4. WAV·대사 대응 검사와 Irodori 목록 생성
settings = dict(captions=str(CAPTIONS), data=str(DATA), minimum=min_audio_seconds,
                maximum=max_audio_seconds, zip_sha256=source_hash)
settings_path = RUN / 'dataset_settings.json'
settings_path.write_text(json.dumps(settings), encoding='utf-8')
prepare_rows = write_script('prepare_rows.py', r"""import csv, json, hashlib, sys
from pathlib import Path
import soundfile as sf
s = json.loads(Path(sys.argv[1]).read_text())
root = Path(s['captions']).parent
data = Path(s['data'])
accepted, excluded, seen = [], [], set()
seconds = 0
with open(s['captions'], encoding='utf-8-sig', newline='') as f:
    reader = csv.DictReader(f)
    if not {'audio_file', 'text'}.issubset(reader.fieldnames or []):
        raise ValueError('captions.csv에는 audio_file,text 열이 필요합니다.')
    for row in reader:
        name, text = row['audio_file'], row['text'].strip()
        audio = (root / 'wav' / name).resolve()
        if not audio.is_relative_to((root / 'wav').resolve()) or not audio.is_file():
            raise ValueError(f'누락/잘못된 WAV 경로: {name}')
        if name in seen:
            raise ValueError(f'중복 오디오 행: {name}')
        seen.add(name)
        info = sf.info(audio)
        reason = ('empty_text' if not text else
                  'too_short' if info.duration < s['minimum'] else
                  'too_long' if info.duration > s['maximum'] else '')
        if reason:
            excluded.append(dict(audio_file=name, seconds=info.duration, reason=reason))
            continue
        accepted.append(dict(audio=str(audio), text=text, speaker='YShirakawa'))
        seconds += info.duration
if len(accepted) < 20:
    raise ValueError(f'학습 가능한 행이 너무 적습니다: {len(accepted)}')
payload = ''.join(json.dumps(r, ensure_ascii=False) + '\n' for r in accepted)
digest = hashlib.sha256(payload.encode()).hexdigest()
metadata = data / 'metadata.jsonl'
if metadata.exists() and metadata.read_text(encoding='utf-8') != payload:
    raise RuntimeError('같은 run의 데이터 조건이 바뀌었습니다. 새 run_name을 사용하세요.')
metadata.write_text(payload, encoding='utf-8')
(data / 'excluded.json').write_text(json.dumps(excluded, ensure_ascii=False, indent=2), encoding='utf-8')
report = dict(total=len(seen), accepted=len(accepted), excluded=len(excluded),
              hours=seconds/3600, metadata_sha256=digest)
(data / 'dataset_report.json').write_text(json.dumps(report, indent=2))
print(json.dumps(report, ensure_ascii=False, indent=2))
print('긴 WAV는 자르지 않고 이 학습 목록에서 제외했습니다. 원본 파일은 유지됩니다.')
""")
run([PY, prepare_rows, settings_path], cwd=REPO)


{
  "total": 4124,
  "accepted": 4032,
  "excluded": 92,
  "hours": 4.199790690350213,
  "metadata_sha256": "958cd3ac6a4166ef7c398082bfae274300ca0788c1fa1c8fa95b9251cfe1c4b5"
}
긴 WAV는 자르지 않고 이 학습 목록에서 제외했습니다. 원본 파일은 유지됩니다.


In [9]:
#@title 5. RF 기본 모델 다운로드 (로컬 캐시)
MODEL_DIR = WORK / 'models/irodori-v4.1-small-rf'
download = write_script('download_model.py', """import sys
from huggingface_hub import snapshot_download
snapshot_download(repo_id=sys.argv[1], revision=sys.argv[2], local_dir=sys.argv[3],
                  allow_patterns=["model.safetensors", "tokenizer/*", "README.md"])
""")
run([PY, download, MODEL_ID, MODEL_REVISION, MODEL_DIR], cwd=REPO)
BASE = MODEL_DIR / 'model.safetensors'
assert BASE.is_file()



Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.

Fetching 4 files: 100%|██████████| 4/4 [00:17<00:00,  4.40s/it]


In [10]:
#@title 6. WAV → DACVAE latent 전처리
MANIFEST = DATA / 'train_manifest.jsonl'
DONE = DATA / 'latents.complete.json'
if not DONE.exists():
    if MANIFEST.exists() or (DATA / 'latents').exists():
        raise RuntimeError('미완료 전처리 결과가 있습니다. 새 run_name으로 시작하세요.')
    run([PY, 'prepare_manifest.py', '--dataset', 'json',
         '--data-files', DATA / 'metadata.jsonl', '--split', 'train',
         '--audio-column', 'audio', '--text-column', 'text',
         '--speaker-column', 'speaker', '--speaker-id-prefix', 'yuina',
         '--output-manifest', MANIFEST, '--latent-dir', DATA / 'latents',
         '--device', 'cuda'], cwd=REPO, log=RUN / 'preprocess.log')
validate = write_script('validate_latents.py', r"""import json, sys
from pathlib import Path
import torch
data = Path(sys.argv[1])
expected = len((data / 'metadata.jsonl').read_text(encoding='utf-8').splitlines())
rows = [json.loads(x) for x in (data / 'train_manifest.jsonl').read_text(encoding='utf-8').splitlines()]
assert len(rows) == expected, f'전처리 누락: 입력 {expected}, 출력 {len(rows)}. preprocess.log를 확인하세요.'
maximum = 0
for row in rows:
    p = (data / row['latent_path']).resolve()
    assert p.is_relative_to(data.resolve()), 'latent 경로 이탈'
    x = torch.load(p, map_location='cpu', weights_only=True)
    assert x.ndim == 2 and x.shape[1] == 32 and x.shape[0] == row['num_frames']
    assert torch.isfinite(x).all() and x.shape[0] > 0
    maximum = max(maximum, x.shape[0])
report = {'rows':len(rows), 'max_latent_steps':maximum}
(data / 'latents.complete.json').write_text(json.dumps(report))
print(report)
""")
run([PY, validate, DATA], cwd=REPO)
LATENT_INFO = json.loads(DONE.read_text())



Generating train split: 0 examples [00:00, ? examples/s]
Generating train split: 4032 examples [00:00, 129366.48 examples/s]
/content/irodori_work/Irodori-TTS/.venv/lib/python3.12/site-packages/audiotools/core/audio_signal.py:44: SyntaxWarning: invalid escape sequence '\_'
  Type of window to use, by default ``sqrt\_hann``.
/content/irodori_work/Irodori-TTS/.venv/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1014: SyntaxWarning: invalid escape sequence '\_'
  using functools.lru\_cache.
/content/irodori_work/Irodori-TTS/.venv/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1092: SyntaxWarning: invalid escape sequence '\_'
  """Compute how the STFT should be padded, based on match\_stride.
/content/irodori_work/Irodori-TTS/.venv/lib/python3.12/site-packages/audiotools/core/audio_signal.py:1141: SyntaxWarning: invalid escape sequence '\_'
  Type of window to use, by default ``sqrt\_hann``.
/content/irodori_work/Irodori-TTS/.venv/lib/python3.12/site-packages/a

**학습 설정:** batch 1, gradient accumulation 8, gradient checkpointing을 기본으로 사용합니다.
훈련용 참조 음성은 최대 10초로 줄여 메모리를 절약합니다. 학습 대상 음성의 latent는 검증된 최대 길이를 설정해 잘림을 방지합니다.
검증용 비율은 2%입니다. 같은 화자의 유사 대사가 섞일 수 있으므로 검증 손실만으로 최종 음질을 판단하지 말고, 학습에 없는 문장도 청취하세요.


In [12]:
#@title 임시 학습 HP 변경
batch_size = 4 #@param {type:"integer"}
gradient_accumulation_steps = 2 #@param {type:"integer"}
learning_rate = 0.0001 #@param {type:"number"}
max_steps = 1000 #@param {type:"integer"}
save_every = 250 #@param {type:"integer"}
lora_rank = 16 #@param {type:"integer"}
reference_max_seconds = 10.0 #@param {type:"number"}

print(f"배치 {batch_size} × 누적 {gradient_accumulation_steps}"
      f" = 유효 배치 {batch_size * gradient_accumulation_steps}")
print(f"학습률 {learning_rate} / {max_steps} steps")

배치 4 × 누적 2 = 유효 배치 8
학습률 0.0001 / 1000 steps


In [13]:
#@title 7. 학습 실행 (선택: smoke_test로 2 step 시험)
smoke_test = False #@param {type:"boolean"}
resume_checkpoint = "" #@param {type:"string"}

# 추론 서버가 GPU를 점유한 상태에서 학습을 시작하지 않습니다.
if globals().get('app_proc') is not None and app_proc.poll() is None:
    raise RuntimeError('마지막 서버 종료 셀을 실행한 뒤 학습하세요.')
target_output = RUN / 'smoke_checkpoints' if smoke_test else OUTPUT
config_path = RUN / ('smoke.yaml' if smoke_test else 'train.yaml')
steps = 2 if smoke_test else max_steps
overrides = dict(
    train_mode='rf', batch_size=batch_size,
    gradient_accumulation_steps=1 if smoke_test else gradient_accumulation_steps,
    num_workers=2, dataloader_prefetch_factor=2, dataloader_cuda_prefetch=False,
    precision=GPU['precision'], allow_tf32=GPU['precision']=='bf16',
    gradient_checkpointing=True, compile_model=False,
    learning_rate=learning_rate, max_steps=steps,
    warmup_steps=0 if smoke_test else min(100, max(1, steps//10)),
    stable_steps=0 if smoke_test else max(0, int(steps*0.8)-min(100, max(1, steps//10))),
    save_every=2 if smoke_test else save_every,
    valid_every=2 if smoke_test else save_every, valid_ratio=0.02,
    checkpoint_best_n=1, log_every=1 if smoke_test else 10,
    max_latent_steps=LATENT_INFO['max_latent_steps'], max_text_len=512,
    ref_min_seconds=1.0, ref_max_seconds=reference_max_seconds,
    lora_enabled=True, lora_r=lora_rank, lora_alpha=2*lora_rank,
    wandb_enabled=False, seed=seed,
)
override_path = RUN / 'overrides.json'
override_path.write_text(json.dumps(overrides))
make_config = write_script('make_config.py', """import json, sys, yaml
from pathlib import Path
cfg = yaml.safe_load(Path(sys.argv[1]).read_text())
cfg['train'].update(json.loads(Path(sys.argv[2]).read_text()))
Path(sys.argv[3]).write_text(yaml.safe_dump(cfg, sort_keys=False))
""")
run([PY, make_config, REPO / 'configs/train_v4_small_lora.yaml', override_path, config_path])
args = [PY, 'train.py', '--config', config_path, '--manifest', MANIFEST,
        '--output-dir', target_output, '--init-checkpoint', BASE, '--device', 'cuda']
if resume_checkpoint.strip():
    if smoke_test:
        raise ValueError('시험 실행에서는 resume_checkpoint를 비우세요.')
    resume = Path(resume_checkpoint).resolve()
    if not resume.is_relative_to(RUN.resolve()) or not (resume / 'adapter_config.json').is_file():
        raise ValueError('현재 로컬 run 안의 LoRA 체크포인트 폴더를 지정하세요.')
    args += ['--resume', resume]
elif target_output.exists() and any(target_output.iterdir()):
    raise RuntimeError('기존 학습 결과가 있습니다. 본 학습은 resume_checkpoint를 지정하거나 run_name을 바꾸세요.')
run(args, cwd=REPO, log=RUN / ('smoke.log' if smoke_test else 'train.log'))
print('시험 완료. smoke_test를 해제하고 이 셀을 다시 실행하세요.' if smoke_test else '본 학습 완료.')
print('결과:', target_output)


Loaded config: /content/irodori_work/runs/yuina_rf_lora/train.yaml
TF32 matmul/cuDNN: enabled
Compute precision=bf16 (weights/optimizer states kept in fp32).
Text tokenizer=sbintuitions/modernbert-ja-310m vocab=102400 add_bos=True padding_side=right (pretrained hidden_size=768).
Caption tokenizer=sbintuitions/modernbert-ja-310m vocab=102400 add_bos=True padding_side=right (pretrained hidden_size=768).
Loaded manifest index cache: /content/irodori_work/runs/yuina_rf_lora/data/train_manifest.jsonl.irodori_index.pt
Reference concat enabled: ref_min_seconds=1.0 ref_max_seconds=10.0 (frames 25..250 at 25 Hz).
Validation split enabled: train=3952 valid=80 (ratio=0.0200, valid_every=250 steps).
Using stratified logit-normal timestep sampling.
Length bucket sampling enabled: window_batches=64 window_samples=256 with shuffled batch order.
Checkpoint retention: latest=1 + best_val_loss=1.
Initialized model weights from: /content/irodori_work/models/irodori-v4.1-small-rf/model.safetensors
LoRA en

**추론 연결:** 다음 셀에서 체크포인트를 선택합니다. 비워 두면 본 학습의 `checkpoint_final` 또는 가장 최근 저장본을 사용합니다.
공식 Gradio UI의 **LoRA Adapter Directory**에 선택한 경로를 자동 입력합니다. 참조 WAV를 함께 올리고 일본어 문장으로 음색을 확인하세요.
학습이 중단되어도 완전히 저장된 체크포인트가 있으면 추론할 수 있습니다.

**터널 선택:** `cloudflared_quick`은 계정 없이 시도할 수 있지만 Cloudflare가 SSE를 공식 지원하지 않아 Gradio 생성이 멈출 수 있습니다.
이때 `gradio_share`로 바꿔 재실행하거나 `cloudflared_named`를 사용하세요. Named Tunnel은 Cloudflare에서 Public Hostname의 서비스 URL을 `http://127.0.0.1:7860`으로 설정하고,
코랩 Secrets에 `CLOUDFLARE_TUNNEL_TOKEN`을 등록합니다. 로그인 비밀번호는 입력창으로 받아 노트북 소스에 저장하지 않습니다.


In [17]:
#@title 8. 추론 체크포인트 선택과 Gradio 시작
lora_checkpoint = "" #@param {type:"string"}
tunnel_mode = "gradio_share" #@param ["cloudflared_quick", "cloudflared_named", "gradio_share"]
named_tunnel_url = "" #@param {type:"string"}
gradio_username = "yuina" #@param {type:"string"}

import getpass
if globals().get('app_proc') is not None and app_proc.poll() is None:
    raise RuntimeError('서버 종료 셀을 먼저 실행하세요.')
if globals().get('tunnel_proc') is not None and tunnel_proc.poll() is None:
    raise RuntimeError('터널 종료 셀을 먼저 실행하세요.')
def valid_adapter(p):
    return (p / 'adapter_config.json').is_file() and (p / 'adapter_model.safetensors').is_file()
if lora_checkpoint.strip():
    ADAPTER = Path(lora_checkpoint).resolve()
else:
    choices = [p for p in OUTPUT.glob('checkpoint_*') if p.is_dir() and valid_adapter(p)]
    if not choices:
        raise RuntimeError('본 학습 체크포인트가 없습니다. 7번 셀에서 본 학습을 실행하세요.')
    final = OUTPUT / 'checkpoint_final'
    ADAPTER = final if valid_adapter(final) else max(choices, key=lambda p:p.stat().st_mtime)
if not ADAPTER.is_relative_to(RUN.resolve()) or not valid_adapter(ADAPTER):
    raise ValueError('현재 run 안의 완전히 저장된 LoRA 폴더를 지정하세요.')
assert gradio_username.strip()
password = getpass.getpass('Gradio 로그인 비밀번호 (8자 이상): ')
if len(password) < 8:
    raise ValueError('비밀번호는 8자 이상 입력하세요.')
wrapper = write_script('launch_gradio.py', r"""import os, sys
from pathlib import Path
sys.path.insert(0, os.environ['IRODORI_REPO'])
import gradio_app as app
app._default_checkpoint = lambda: os.environ['IRODORI_BASE']
demo = app.build_ui()
adapters = [c for c in demo.blocks.values() if getattr(c, 'label', '') == 'LoRA Adapter Directory (optional)']
assert len(adapters) == 1, '공식 UI의 LoRA 입력란이 변경되었습니다.'
selected = os.environ['IRODORI_ADAPTER']
with demo:
    demo.load(lambda: selected, inputs=None, outputs=adapters[0], queue=False)
demo.queue(default_concurrency_limit=1, max_size=8)
demo.launch(server_name='127.0.0.1', server_port=7860,
    share=os.environ['IRODORI_SHARE']=='1',
    auth=(os.environ.pop('IRODORI_USER'), os.environ.pop('IRODORI_PASSWORD')),
    blocked_paths=['/content/drive', str(Path.home() / '.config'),
                   os.environ['HF_HOME'], os.environ['IRODORI_CHECKPOINT_ROOT']],
    css=app.EMOJI_PALETTE_CSS)
""")
app_env = os.environ.copy()
app_env.update(IRODORI_REPO=str(REPO), IRODORI_BASE=str(BASE), IRODORI_ADAPTER=str(ADAPTER),
               IRODORI_SHARE='1' if tunnel_mode=='gradio_share' else '0',
               IRODORI_USER=gradio_username, IRODORI_PASSWORD=password,
               IRODORI_CHECKPOINT_ROOT=str(OUTPUT))
app_log = RUN / 'gradio.log'
with open(app_log, 'w') as stream:
    app_proc = subprocess.Popen([str(PY), '-u', str(wrapper)], cwd=REPO, env=app_env,
                                stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
app_env.pop('IRODORI_PASSWORD', None)
del password
for _ in range(120):
    if app_proc.poll() is not None:
        raise RuntimeError(app_log.read_text()[-8000:])
    try:
        urllib.request.urlopen('http://127.0.0.1:7860/', timeout=2).close()
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Gradio 시작 시간 초과. gradio.log를 확인하고 서버 종료 셀을 실행하세요.')
print('Gradio 준비 완료. LoRA:', ADAPTER)
print('참조 WAV 예시:', next((CAPTIONS.parent / 'wav').glob('*.wav')))


Gradio 로그인 비밀번호 (8자 이상): ··········
Gradio 준비 완료. LoRA: /content/irodori_work/runs/yuina_rf_lora/checkpoints/checkpoint_final
참조 WAV 예시: /content/irodori_work/runs/yuina_rf_lora/dataset/yuina/wav/VC_Common_00009_v008_YShirakawa.wav


In [18]:
#@title 9. 외부 접속 주소 열기 (cloudflared / Gradio 공유)
from IPython.display import display, Markdown
if app_proc.poll() is not None:
    raise RuntimeError('Gradio가 종료되었습니다. 8번 셀을 확인하세요.')
if tunnel_mode == 'gradio_share':
    for _ in range(60):
        found = re.search(r'https://[a-zA-Z0-9-]+\.gradio\.live', app_log.read_text())
        if found:
            break
        if app_proc.poll() is not None:
            raise RuntimeError(app_log.read_text()[-4000:])
        time.sleep(1)
    if not found:
        raise RuntimeError('공유 주소 생성 실패. gradio.log를 확인하세요.')
    public_url = found.group(0)
else:
    if globals().get('tunnel_proc') is not None and tunnel_proc.poll() is None:
        raise RuntimeError('터널이 이미 실행 중입니다. 종료 셀을 먼저 실행하세요.')
    binary = WORK / 'cloudflared'
    cloud_version = '2026.9.1'
    cloud_sha256 = '03f1f25d1cc93b9ad6c60569d44060bc4f17ed97075760ed8cfca4b12dcd68cc'
    if not binary.exists():
        urllib.request.urlretrieve(
            f'https://github.com/cloudflare/cloudflared/releases/download/{cloud_version}/cloudflared-linux-amd64', binary)
    if sha256(binary) != cloud_sha256:
        raise RuntimeError('cloudflared 바이너리 SHA256 불일치')
    binary.chmod(0o700)
    tunnel_env = os.environ.copy()
    if tunnel_mode == 'cloudflared_named':
        from google.colab import userdata
        if not re.fullmatch(r'https://[A-Za-z0-9.-]+/?', named_tunnel_url.strip()):
            raise ValueError('Cloudflare에 설정한 HTTPS 호스트 주소를 입력하세요.')
        tunnel_env['TUNNEL_TOKEN'] = userdata.get('CLOUDFLARE_TUNNEL_TOKEN')
        args = [binary, 'tunnel', '--no-autoupdate', 'run']
    else:
        args = [binary, 'tunnel', '--no-autoupdate', '--url', 'http://127.0.0.1:7860']
    tunnel_log = RUN / 'cloudflared.log'
    with open(tunnel_log, 'w') as stream:
        tunnel_proc = subprocess.Popen([str(a) for a in args], env=tunnel_env,
                                       stdout=stream, stderr=subprocess.STDOUT, start_new_session=True)
    tunnel_env.pop('TUNNEL_TOKEN', None)
    for _ in range(60):
        if tunnel_proc.poll() is not None:
            raise RuntimeError('cloudflared 종료. 로컬 cloudflared.log를 확인하세요.')
        log_text = tunnel_log.read_text()
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', log_text)
        if tunnel_mode == 'cloudflared_named' and 'Registered tunnel connection' in log_text:
            public_url = named_tunnel_url.strip()
            break
        if tunnel_mode == 'cloudflared_quick' and found:
            public_url = found.group(0)
            break
        time.sleep(1)
    else:
        raise RuntimeError('터널 연결 시간 초과. 종료 셀 실행 후 재시도하세요.')
display(Markdown(f'**접속:** [{public_url}]({public_url}) · 로그인 ID: `{gradio_username}`'))
print('LoRA Adapter Directory:', ADAPTER)
if tunnel_mode == 'cloudflared_quick':
    print('Quick Tunnel에서 생성이 멈추면 서버 종료 → tunnel_mode=gradio_share → 8~9번 재실행.')


**접속:** [https://b03425eeb1837582e8.gradio.live](https://b03425eeb1837582e8.gradio.live) · 로그인 ID: `yuina`

LoRA Adapter Directory: /content/irodori_work/runs/yuina_rf_lora/checkpoints/checkpoint_final


**Gradio 사용 순서:** `LoRA Adapter Directory` 경로 확인 → 유이나 참조 WAV 업로드 → 일본어 문장 입력 → Generate.
모델은 첫 생성 시 로드됩니다. 다른 체크포인트를 비교하려면 같은 run 안의 LoRA 폴더 경로를 UI에 입력하세요.
메모리가 부족하면 후보 개수를 1로 유지하고 짧은 문장/짧은 참조 음성으로 먼저 확인합니다.


In [19]:
#@title 10. 선택한 LoRA를 PC로 다운로드 (Drive 저장 없음)
download_lora = True #@param {type:"boolean"}
include_resume_state = True #@param {type:"boolean"}
if download_lora:
    from google.colab import files
    if not globals().get('ADAPTER') or not valid_adapter(ADAPTER):
        raise RuntimeError('8번 셀에서 체크포인트를 먼저 선택하세요.')
    destination = RUN / (ADAPTER.name + ('_resume.zip' if include_resume_state else '_inference.zip'))
    with zipfile.ZipFile(destination, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as z:
        for p in ADAPTER.rglob('*'):
            if p.is_file() and (include_resume_state or p.suffix != '.pt'):
                z.write(p, ADAPTER.name + '/' + p.relative_to(ADAPTER).as_posix())
        z.writestr('BASE_MODEL.json', json.dumps(dict(repo_id=MODEL_ID, revision=MODEL_REVISION,
                    code_revision=CODE_REVISION), indent=2))
    print('PC 다운로드:', destination)
    files.download(str(destination))
else:
    print('보관하려면 download_lora를 체크하고 실행하세요. 런타임 종료 시 로컬 결과가 사라집니다.')


PC 다운로드: /content/irodori_work/runs/yuina_rf_lora/checkpoint_final_resume.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [16]:
#@title 11. Gradio·터널 종료 (체크포인트는 유지)
for variable in ('tunnel_proc', 'app_proc'):
    process = globals().get(variable)
    if process is not None and process.poll() is None:
        os.killpg(process.pid, signal.SIGTERM)
        try:
            process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            os.killpg(process.pid, signal.SIGKILL)
            process.wait()
        print(variable, '종료 완료')
print('저장된 결과:', OUTPUT)


tunnel_proc 종료 완료
app_proc 종료 완료
저장된 결과: /content/irodori_work/runs/yuina_rf_lora/checkpoints
